# GameTheory-16h : Coopération entre agents — protocole multi-seed, jumeau C#

**Version C#** du carnet [GameTheory-16h — Coopération LLM](GameTheory-16h-Cooperation-LLM-Python.ipynb). Même numéro, même titre, même protocole : le carnet rejoue un **Dilemme du Prisonnier Itéré** (IPD, Axelrod 1984) entre agents **hétérogènes**, sous un protocole gelé avant le premier tour, et mesure si la composition de la population change le taux de coopération.

Le LLM n'est pas appelé : il est représenté par un **proxy scripté** (`generous_tft`), calibré sur Vallinder et Hughes (2024). Ce que le carnet mesure est le **protocole** et le contraste entre hétérogénéités, pas le comportement d'un modèle de langage.

**Ce que ce jumeau ajoute au carnet Python.** Le carnet Python tire ses suites avec le générateur MT19937 de `random.Random` ; la bibliothèque standard .NET ne le reconstruit pas. Le jumeau tire donc ses **propres** suites avec `new Random(seed)` sur les **cinq mêmes graines gelées**, et il n'affirme aucune valeur numérique du Python : il affirme les **lois** que la source lit, par des assertions exécutables — bande de nullité du modèle nul, zéro strict du contrôle négatif, coopération totale des deux hétérogénéités à proxy, verdict global. La section « Aléa » du §4 dit laquelle des deux lectures s'applique à chaque nombre affiché.

**Objectifs** : geler un protocole avant exécution, exécuter un tournoi IPD multi-seed sur quatre hétérogénéités, valider la chaîne par un modèle nul et un contrôle négatif, rendre un verdict H0/H1 par hétérogénéité.

**Prérequis** : programmation C# de base (délégués, dictionnaires, tuples), probabilités élémentaires, notion de test d'hypothèse.

**Durée estimée** : 45 minutes.

**Navigation** : [Version Python](GameTheory-16h-Cooperation-LLM-Python.ipynb) | [Index GameTheory](README.md) | Précédent : [GameTheory-16e — Populations Othman-Sandholm](GameTheory-16e-Populations-Othman-Sandholm-Python.ipynb) | Suivant : [GameTheory-16i — Appariage stable (Gale-Shapley)](GameTheory-16i-Appariage-Stable-Gale-Shapley-CSharp.ipynb) | Pilote : [GameTheory-16d — Joueurs LLM hétérogènes](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb).


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Le noyau .NET Interactive execute chaque cellule sur un thread potentiellement
// different : CurrentCulture seul ne suit pas d'une cellule a l'autre. La culture
// par defaut des threads est posee ici pour que tous les nombres s'affichent au
// point decimal, quel que soit le thread qui joue la cellule.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

// =====================================================================
// GEL §0 — protocole fige AVANT tout premier appel d'agent
// =====================================================================
var SEEDS = new List<int> { 0, 1, 7, 42, 99 };   // sous-ensemble strict du jeu standard (>= 4)
var N_ROUNDS = 100;                              // longueur IPD (Axelrod 1984)
// RESERVE, non utilise : le protocole apparie UNE paire de politiques par heterogeneite
// (2 agents par rencontre, N_MATCHES_PER_PAIR rencontres). Le symbole reste declare pour
// qu'une reprise "populations multiples par paire" le trouve, et il est imprime avec sa
// mention pour qu'un lecteur du gel ne croie pas qu'il est joue.
var N_POPULATIONS_PER_PAIR = 2;                  // RESERVE -- non utilise par ce protocole
var N_MATCHES_PER_PAIR = 50;                     // rencontres independantes par (heterogeneite, seed)

void Check(bool condition, string message)
{
    if (!condition)
        throw new InvalidOperationException(message);
}

Check(SEEDS.Count >= 4, "Multi-seed protocol : >= 4 seeds parmi {0,1,7,42,99}.");
Check(N_ROUNDS >= 50, "IPD rounds : N_ROUNDS >= 50 pour stabilite.");
Check(N_MATCHES_PER_PAIR >= 30, "Sample size : N_MATCHES_PER_PAIR >= 30 pour CLT.");

Console.WriteLine("Imports + gel preliminaire OK.");
Console.WriteLine($"  SEEDS = [{string.Join(", ", SEEDS)}]");
Console.WriteLine($"  N_ROUNDS = {N_ROUNDS}, N_MATCHES_PER_PAIR = {N_MATCHES_PER_PAIR}");
Console.WriteLine($"  N_POPULATIONS_PER_PAIR = {N_POPULATIONS_PER_PAIR} (reserve, non utilise)");

Imports + gel preliminaire OK.


  SEEDS = [0, 1, 7, 42, 99]


  N_ROUNDS = 100, N_MATCHES_PER_PAIR = 50


  N_POPULATIONS_PER_PAIR = 2 (reserve, non utilise)


## Préambule — qu'est-ce qu'une *hétérogénéité de population* en IPD ?

Le pilote `GameTheory-16d` joue **une paire** (row, col) sur un mécanisme ; le `GameTheory-16e` définit une **population** comme une distribution sur les types. Ici, on change de jeu : on joue un **Dilemme du Prisonnier Itéré** et une *hétérogénéité* est une **distribution sur les stratégies** des agents qui peuplent la rencontre.

Quatre hétérogénéités sont définies au §2 :

- **`llm_only`** : deux agents LLM, l'un TFT généreux, l'autre généreux TFT. Modélisation CPU : un *proxy* de politique scriptée qui imite le comportement LLM dominant mesuré par Vallinder–Hughes 2024.
- **`mix_llm_scripted`** : un agent LLM (généreux) et un agent scripté (`tit_for_tat` strict). Modélise la **coopération inter-classes**.
- **`scripted_only`** : deux agents scriptés (généreux TFT contre TFT suspect). Baseline de **coopération purement scriptée**.
- **`always_defect`** : deux agents `always_defect`. **Contrôle négatif** : doit dégrader la coopération à 0 %.

Le **modèle nul** `uniform_random_seed` (§5) sert de baseline inerte : chaque agent joue `C` ou `D` uniformément au hasard, indépendamment du passé — c'est la **strate inférieure** de la coopération, et c'est à lui que le §7 compare chaque hétérogénéité.


## 0. Pré-enregistrement du protocole (gelé avant tout appel d'agent)

Rôles, conditions, hétérogénéités, graines, hypothèses, seuils de verdict et contrôles sont figés ici. Toute déviation exécution contre gel est consignée à la section « Écarts d'exécution vs gel ».

**Rôles** : `agent_0` (joueur 1) et `agent_1` (joueur 2) — symétrie d'information.

**Jeu** : Dilemme du Prisonnier Itéré, payoff `(T, R, P, S) = (5, 3, 1, 0)`. Axelrod 1984, tournoi informatique.

**Hétérogénéités (4, dont 1 contrôle négatif)** :

- `llm_only` : deux agents LLM, (généreux TFT, généreux TFT).
- `mix_llm_scripted` : un agent LLM généreux et un TFT strict. Hétérogénéité **inter-classes**.
- `scripted_only` : deux scriptés, (généreux TFT, TFT suspect).
- `always_defect` : deux agents `always_defect`. **Contrôle négatif** — doit dégrader à 0 % de coopération.

**Politique d'information** : un agent observe l'historique complet des coups joués par l'adversaire (IPD canonique), sans information cachée.

**Graines** : `{0, 1, 7, 42, 99}` — sous-ensemble strict de 4+ éléments du jeu standard. Cinq graines pour la robustesse.

**Métriques** :

- `cooperation_rate(match)` : proportion de tours où l'agent joue `C`.
- `mean_coop_rate(het, seed)` : moyenne sur les `N_MATCHES_PER_PAIR` rencontres indépendantes.
- `total_payoff` : somme des payoffs de l'agent sur les `N_ROUNDS` tours (mesure secondaire).

**Seuils de verdict (H0/H1)** :

- **H0** — la coopération moyenne est constante entre hétérogénéités : moyenne par graine dans `± 0.05` du modèle nul.
- **H1** — au moins une hétérogénéité sort de la bande `± 0.05`.

**Modèle nul** : `uniform_random_seed` — chaque tour, l'agent joue `C` ou `D` avec probabilité 0.5.

**Contrôle négatif** : `always_defect` — l'agent joue `D` à tous les tours ; le taux de coopération doit être **exactement 0**, et le payoff mutuel `P × N_ROUNDS = 100`.


In [2]:
// §0 — gel du protocole (assertions d'auto-coherence)
Check(SEEDS.SequenceEqual(new[] { 0, 1, 7, 42, 99 }), "Seeds gelees.");
Check(N_ROUNDS == 100, "N gele.");
Check(N_MATCHES_PER_PAIR == 50, "N_MATCHES gele.");

var HETEROGENEITIES = new List<string> { "llm_only", "mix_llm_scripted", "scripted_only", "always_defect" };
Check(HETEROGENEITIES.Count >= 2, "Au moins 2 heterogeneites (4 retenues).");
var NULL_MODEL = "uniform_random_seed";   // baseline inerte (pas dans HETEROGENEITIES, sert au §5)
var NEGATIVE_CONTROL = "always_defect";
Check(HETEROGENEITIES.Contains(NEGATIVE_CONTROL), "Le controle negatif est dans les heterogeneites.");
var H0_NULL_BAND = 0.05;                  // moyenne dans +/- 0.05 du nul => H0

Console.WriteLine("§0 protocole gele :");
Console.WriteLine($"  SEEDS = [{string.Join(", ", SEEDS)}]");
Console.WriteLine($"  N_ROUNDS = {N_ROUNDS}, N_MATCHES_PER_PAIR = {N_MATCHES_PER_PAIR}, N_POPULATIONS_PER_PAIR = {N_POPULATIONS_PER_PAIR} (reserve, non utilise)");
Console.WriteLine($"  HETEROGENEITIES = [{string.Join(", ", HETEROGENEITIES)}]");
Console.WriteLine($"  NULL_MODEL = {NULL_MODEL} (modele nul, baseline §5)");
Console.WriteLine($"  NEGATIVE_CONTROL = {NEGATIVE_CONTROL}");
Console.WriteLine($"  H0 band = +/- {H0_NULL_BAND}");


§0 protocole gele :


  SEEDS = [0, 1, 7, 42, 99]


  N_ROUNDS = 100, N_MATCHES_PER_PAIR = 50, N_POPULATIONS_PER_PAIR = 2 (reserve, non utilise)


  HETEROGENEITIES = [llm_only, mix_llm_scripted, scripted_only, always_defect]


  NULL_MODEL = uniform_random_seed (modele nul, baseline §5)


  NEGATIVE_CONTROL = always_defect


  H0 band = +/- 0.05


## 1. Le jeu IPD et les politiques (proxy scripté des agents LLM)

Le **Dilemme du Prisonnier Itéré** est le substrat canonique des études de coopération (Axelrod 1984). Le payoff par tour est `(T, R, P, S) = (5, 3, 1, 0)` : la condition `T > R > P > S` et `2R > T + S` garantit la **structure du dilemme** — la défection individuelle est tentante, mais la coopération mutuelle répétée est collectivement meilleure.

**Politiques scriptées** (proxy des agents LLM, Vallinder–Hughes 2024) :

- **`tft`** : coopère au tour 1, puis copie le dernier coup de l'adversaire.
- **`generous_tft`** : TFT, mais avec probabilité `p_gen = 0.1` de **pardonner** une défection.
- **`suspicious_tft`** : défection au tour 1, puis TFT.
- **`always_defect`** : joue `D` à tous les tours. **Contrôle négatif**.
- **`uniform_random_seed`** : joue `C` ou `D` uniformément au hasard. **Modèle nul**.
- **`llm_proxy_tft`** : le proxy d'un agent LLM « qui coopère », c'est-à-dire `generous_tft`.

**Hypothèse de modélisation** : un LLM *prompté* à coopérer joue en moyenne comme un `generous_tft` (Vallinder–Hughes 2024 : les LLM montrent une générosité significativement supérieure à un TFT strict dans le tournoi IPD). Le proxy est volontairement **fruste** : l'intérêt du protocole est dans la chaîne reproductible, pas dans la finesse de la politique LLM.


In [3]:
// §1 — Jeu IPD et politiques (proxy scripte des agents LLM)
var T = 5;   // temptation
var R = 3;   // reward
var P = 1;   // punishment
var S = 0;   // sucker
var ACTIONS = new[] { "C", "D" };   // C = cooperate, D = defect

record Round(string A0, string A1);

(int, int) Payoff(string self, string other)
{
    if (self == "C" && other == "C") return (R, R);
    if (self == "C" && other == "D") return (S, T);
    if (self == "D" && other == "C") return (T, S);
    return (P, P);   // both D
}

Func<List<Round>, Random, int, string> TitForTat = (history, rng, role) =>
{
    if (history.Count == 0) return "C";
    var last = history[history.Count - 1];
    return role == 0 ? last.A1 : last.A0;
};

Func<List<Round>, Random, int, string> GenerousTft = (history, rng, role) =>
{
    if (history.Count == 0) return "C";
    var last = history[history.Count - 1];
    var otherLast = role == 0 ? last.A1 : last.A0;
    if (otherLast == "D" && rng.NextDouble() < 0.1) return "C";   // pardon
    return otherLast;
};

Func<List<Round>, Random, int, string> SuspiciousTft = (history, rng, role) =>
{
    if (history.Count == 0) return "D";
    var last = history[history.Count - 1];
    return role == 0 ? last.A1 : last.A0;
};

Func<List<Round>, Random, int, string> AlwaysDefect = (history, rng, role) => "D";

Func<List<Round>, Random, int, string> UniformRandomSeed = (history, rng, role) =>
    rng.Next(ACTIONS.Length) == 0 ? "C" : "D";

// Politique LLM proxy : generous_tft (proxy Vallinder-Hughes 2024)
Func<List<Round>, Random, int, string> LlmProxyTft = GenerousTft;

var POLICIES = new Dictionary<string, Func<List<Round>, Random, int, string>>
{
    { "tft", TitForTat },
    { "generous_tft", GenerousTft },
    { "suspicious_tft", SuspiciousTft },
    { "always_defect", AlwaysDefect },
    { "uniform_random_seed", UniformRandomSeed },
    { "llm_proxy_tft", LlmProxyTft },
};
Check(POLICIES.Count >= 5, $"Au moins 5 politiques (j'en ai {POLICIES.Count}).");
Console.WriteLine($"§1 — {POLICIES.Count} politiques chargees : [{string.Join(", ", POLICIES.Keys)}]");
Console.WriteLine($"  Payoff matrix : T={T}, R={R}, P={P}, S={S}");
Check(T > R && R > P && P > S, "Structure du dilemme : T > R > P > S.");
Check(2 * R > T + S, "Cooperation mutuelle iteree : 2R > T + S.");
Console.WriteLine($"  Structure dilemme : T > R > P > S, 2R > T + S ({2 * R} > {T + S})");


§1 — 6 politiques chargees : [tft, generous_tft, suspicious_tft, always_defect, uniform_random_seed, llm_proxy_tft]


  Payoff matrix : T=5, R=3, P=1, S=0


  Structure dilemme : T > R > P > S, 2R > T + S (6 > 5)


**Lecture — §1.** Six politiques sont définies : trois variantes TFT (strict, généreux, suspect), un `always_defect` (contrôle négatif), un `uniform_random_seed` (modèle nul), et le proxy LLM qui est `generous_tft`. Le registre `POLICIES` les expose par nom, ce qui permettra au §2 de définir une hétérogénéité comme un simple couple de clés.

La **structure du dilemme** est vérifiée par deux assertions exécutables, et non par un commentaire :

- `T > R` (5 > 3) : la défection contre un coopérant rapporte plus que la coopération mutuelle.
- `R > P` (3 > 1) : la coopération mutuelle est meilleure que la défection mutuelle.
- `P > S` (1 > 0) : la défection unilatérale est moins punie que la coopération unilatérale.
- `2R > T + S` (6 > 5) : la coopération mutuelle répétée bat l'alternance exploitation/exploitation. C'est cette dernière condition qui fait de la coopération un équilibre rentable à long terme, et sans elle le tournoi n'aurait pas d'intérêt.


## 2. Définition d'une hétérogénéité — distribution sur (strat_0, strat_1)

Une **hétérogénéité** est un **couple de politiques** `(strat_0, strat_1)` qui déterminent les deux agents de la rencontre IPD. Le gel §0 a listé les quatre hétérogénéités canoniques ; ce §2 les implémente.

**Conventions** :

- `llm_only` = `(llm_proxy_tft, generous_tft)` : deux « agents LLM » (proxy généreux).
- `mix_llm_scripted` = `(llm_proxy_tft, tft)` : un LLM et un scripté strict.
- `scripted_only` = `(generous_tft, suspicious_tft)` : deux scriptés, hétérogènes (générosité contre suspicion).
- `always_defect` = `(always_defect, always_defect)` : contrôle négatif — doit dégrader à 0 %.


In [4]:
// §2 — Definition des heterogeneites (couples de politiques)
// Cles : nom de la politique dans le registre POLICIES (cf §1)
var HET_PAIRS = new Dictionary<string, (string, string)>
{
    { "llm_only", ("llm_proxy_tft", "generous_tft") },
    { "mix_llm_scripted", ("llm_proxy_tft", "tft") },
    { "scripted_only", ("generous_tft", "suspicious_tft") },
    { "always_defect", ("always_defect", "always_defect") },
};
Check(new HashSet<string>(HET_PAIRS.Keys).SetEquals(HETEROGENEITIES), "HET_PAIRS doit couvrir HETEROGENEITIES.");
Console.WriteLine("§2 — Heterogeneites :");
foreach (var kv in HET_PAIRS)
    Console.WriteLine($"  {kv.Key,-20} = ({kv.Value.Item1}, {kv.Value.Item2})");


§2 — Heterogeneites :


  llm_only             = (llm_proxy_tft, generous_tft)


  mix_llm_scripted     = (llm_proxy_tft, tft)


  scripted_only        = (generous_tft, suspicious_tft)


  always_defect        = (always_defect, always_defect)


**Lecture — §2.** Les quatre hétérogénéités sont définies par un couple de clés du registre `POLICIES`. L'assertion d'égalité d'ensembles garantit qu'aucune hétérogénéité du gel §0 n'est oubliée à l'implémentation : c'est le point où un protocole pré-enregistré et son code peuvent silencieusement diverger, et il est fermé mécaniquement.

**`mix_llm_scripted`** est l'hétérogénéité la plus instructive du point de vue de la sélection culturelle (Vallinder–Hughes 2024) : un agent LLM généreux face à un TFT strict. Le TFT strict ne pardonne pas, donc la moindre défection du LLM déclenche un cycle de représailles — la générosité du LLM est ici **pénalisée** par le scripté strict, à l'inverse de la dynamique `llm_only` où les deux partenaires se pardonnent mutuellement. Le §4 mesurera si cette asymétrie se matérialise réellement dans ce jeu, ou si elle reste théorique.


## 3. Driver multi-seed — exécution reproductible d'une rencontre IPD

Pour chaque `(hétérogénéité, graine)`, on tire `N_MATCHES_PER_PAIR` rencontres **indépendantes** de `N_ROUNDS` tours chacune. À chaque tour, chaque agent choisit son coup selon sa politique et l'historique. On accumule le **taux de coopération** de chaque agent (moyenne sur les tours) et le **payoff total**.

**Hypothèse de reproductibilité** : le générateur est local à la rencontre, `new Random(seed)`. Deux exécutions successives du carnet rendent donc exactement les mêmes valeurs — c'est ce qui autorise des assertions de loi sur les sorties.

**Combinateur de graine** : la rencontre `m` de la graine `s` utilise `s * 1000 + m`. Sans lui, les cinquante rencontres d'une même graine tireraient la même suite, et la moyenne par graine aurait la variance d'un seul tirage au lieu de cinquante.


In [5]:
// §3 — Driver multi-seed : IPD multi-match, cooperation et payoff
record MatchResult(double CoopRate0, double CoopRate1, int Payoff0, int Payoff1);

MatchResult RunMatch(string strat0, string strat1, int seed)
{
    var rng = new Random(seed);   // RNG local : determinisme par seed
    var p0 = POLICIES[strat0];
    var p1 = POLICIES[strat1];
    var history = new List<Round>();
    var payoff0 = 0;
    var payoff1 = 0;
    var coop0 = 0;
    var coop1 = 0;
    for (var t = 0; t < N_ROUNDS; t++)
    {
        var a0 = p0(history, rng, 0);
        var a1 = p1(history, rng, 1);
        history.Add(new Round(a0, a1));
        var (u0, u1) = Payoff(a0, a1);
        payoff0 += u0;
        payoff1 += u1;
        if (a0 == "C") coop0++;
        if (a1 == "C") coop1++;
    }
    return new MatchResult((double)coop0 / N_ROUNDS, (double)coop1 / N_ROUNDS, payoff0, payoff1);
}

List<MatchResult> RunHeterogeneity(string hetName, int seed, int? nMatches = null)
{
    var n = nMatches ?? N_MATCHES_PER_PAIR;
    var pair = HET_PAIRS[hetName];
    var rows = new List<MatchResult>();
    for (var m = 0; m < n; m++)
        rows.Add(RunMatch(pair.Item1, pair.Item2, seed * 1000 + m));   // seed combine : (het, seed, m)
    return rows;
}

// Test rapide : une rencontre sur le controle negatif
var testRows = RunHeterogeneity("always_defect", 0, 1);
Check(testRows[0].CoopRate0 == 0.0, "Controle negatif : agent 0 doit cooperer 0%");
Check(testRows[0].CoopRate1 == 0.0, "Controle negatif : agent 1 doit cooperer 0%");
Check(testRows[0].Payoff0 == P * N_ROUNDS, $"Controle negatif : payoff_0 attendu {P * N_ROUNDS}");
Check(testRows[0].Payoff1 == P * N_ROUNDS, $"Controle negatif : payoff_1 attendu {P * N_ROUNDS}");
Console.WriteLine("§3 — Driver OK. Test controle negatif :");
Console.WriteLine($"  always_defect seed=0 : coop_0 = {testRows[0].CoopRate0}, coop_1 = {testRows[0].CoopRate1}");
Console.WriteLine("  (attendu : 0 / 0)");
Console.WriteLine($"  payoff_0 = {testRows[0].Payoff0}, payoff_1 = {testRows[0].Payoff1}");
Console.WriteLine($"  (attendu : {P * N_ROUNDS} / {P * N_ROUNDS} = defection mutuelle)");


§3 — Driver OK. Test controle negatif :


  always_defect seed=0 : coop_0 = 0, coop_1 = 0


  (attendu : 0 / 0)


  payoff_0 = 100, payoff_1 = 100


  (attendu : 100 / 100 = defection mutuelle)


**Lecture — §3.** Le driver prend une hétérogénéité et une graine, et exécute cinquante rencontres indépendantes. Le contrôle négatif est joué **avant** le tournoi complet, sur une seule rencontre : c'est le canari de la chaîne. S'il rendait autre chose que zéro coopération et un payoff de `P × N_ROUNDS = 100`, tout le reste du carnet serait invalide, et l'assertion arrête l'exécution sur place plutôt que de laisser produire un tableau trompeur.

**Coût de calcul** : 4 hétérogénéités × 5 graines × 50 rencontres × 100 tours = **100 000 tours** IPD. C'est très léger — le paragraphe suivant le mesure en une fraction de seconde, sans GPU et sans appel réseau.


## 4. Agrégation par (hétérogénéité, graine) — distribution du taux de coopération

Pour chaque `(hétérogénéité, graine)`, on agrège les `N_MATCHES_PER_PAIR` rencontres en **moyenne et écart-type** du taux de coopération de chaque agent, et en payoff moyen. L'écart-type est l'écart-type **d'échantillon** (division par `n - 1`), celui qui estime la dispersion des rencontres et non celle de la population.


In [6]:
// §4 — Aggregation par (heterogeneite, seed)
record Aggregate(double MeanCoop0, double StdCoop0, double MeanCoop1, double StdCoop1,
                 double MeanPayoff0, double MeanPayoff1, int NMatches);

double Mean(IEnumerable<double> xs)
{
    var l = xs.ToList();
    return l.Sum() / l.Count;
}

double Stdev(IEnumerable<double> xs)
{
    var l = xs.ToList();
    if (l.Count <= 1) return 0.0;
    var mu = Mean(l);
    return Math.Sqrt(l.Sum(x => (x - mu) * (x - mu)) / (l.Count - 1));
}

// Execution complete : 4 heterogeneites x 5 seeds x 50 matches
var resultsByHet = new Dictionary<string, Dictionary<int, List<MatchResult>>>();
foreach (var het in HETEROGENEITIES)
{
    resultsByHet[het] = new Dictionary<int, List<MatchResult>>();
    foreach (var seed in SEEDS)
        resultsByHet[het][seed] = RunHeterogeneity(het, seed);
}

var aggregated = new Dictionary<string, Dictionary<int, Aggregate>>();
foreach (var het in HETEROGENEITIES)
{
    aggregated[het] = new Dictionary<int, Aggregate>();
    foreach (var seed in SEEDS)
    {
        var rows = resultsByHet[het][seed];
        aggregated[het][seed] = new Aggregate(
            Mean(rows.Select(r => r.CoopRate0)), Stdev(rows.Select(r => r.CoopRate0)),
            Mean(rows.Select(r => r.CoopRate1)), Stdev(rows.Select(r => r.CoopRate1)),
            Mean(rows.Select(r => (double)r.Payoff0)), Mean(rows.Select(r => (double)r.Payoff1)),
            rows.Count);
    }
}

Console.WriteLine("§4 — Resultats agreges par (heterogeneite, seed) :");
Console.WriteLine("  " + "het".PadRight(20) + "seed".PadLeft(5) + "mean_c0".PadLeft(11)
                  + "std_c0".PadLeft(9) + "mean_c1".PadLeft(11) + "std_c1".PadLeft(9)
                  + "payoff0".PadLeft(11));
foreach (var het in HETEROGENEITIES)
    foreach (var seed in SEEDS)
    {
        var a = aggregated[het][seed];
        Console.WriteLine($"  {het,-20}{seed,5}{a.MeanCoop0,11:F4}{a.StdCoop0,9:F4}"
                          + $"{a.MeanCoop1,11:F4}{a.StdCoop1,9:F4}{a.MeanPayoff0,11:F2}");
    }


§4 — Resultats agreges par (heterogeneite, seed) :


  het                  seed    mean_c0   std_c0    mean_c1   std_c1    payoff0


  llm_only                0     1.0000   0.0000     1.0000   0.0000     300.00


  llm_only                1     1.0000   0.0000     1.0000   0.0000     300.00


  llm_only                7     1.0000   0.0000     1.0000   0.0000     300.00


  llm_only               42     1.0000   0.0000     1.0000   0.0000     300.00


... (12 outputs left out)

  always_defect           1     0.0000   0.0000     0.0000   0.0000     100.00


  always_defect           7     0.0000   0.0000     0.0000   0.0000     100.00


  always_defect          42     0.0000   0.0000     0.0000   0.0000     100.00


  always_defect          99     0.0000   0.0000     0.0000   0.0000     100.00


**Aléa — ce que ces nombres sont, et ce qu'ils ne sont pas.** Le carnet Python tire ses suites avec le générateur MT19937 de `random.Random` ; la BCL .NET ne le reconstruit pas, et un tirage ne se recopie donc pas. Les chiffres du carnet Python sont des **données propres à son générateur** : le nul y vaut `0.5126`, `0.5056`, `0.4994`, `0.5060`, `0.4954` (moyenne `0.5038`, écart-type inter-graines `0.0066`) et `scripted_only` y vaut `0.9187 ± 0.0104`. Le jumeau C# **ne les recopie pas comme un résultat de parité** : il tire ses propres suites avec `new Random(seed)` sur les cinq mêmes graines, et il affirme les **lois** que les deux carnets partagent — bande de nullité autour de 0.5, zéro strict du contrôle négatif, coopération totale des hétérogénéités à proxy, verdict global. Les valeurs affichées ci-dessus sont celles du générateur .NET, exactes et reproductibles d'une exécution à l'autre.


**Lecture — §4.** Le tableau se lit en trois régimes, qui ne dépendent pas du générateur : ils sont structurés par les politiques. Les deux premières lignes forment le même régime.

1. **`llm_only`** : coopération à **1.0000** sur les cinq graines, écart-type nul. Les deux agents généreux coopèrent au premier tour et se copient ensuite : aucune défection n'est jamais jouée, donc la branche de pardon de `generous_tft` n'est même pas tirée. C'est un point fixe, pas un résultat statistique — l'écart-type nul le dit.
2. **`mix_llm_scripted`** : coopération à **1.0000** sur les cinq graines, pour la même raison mécanique. Le mécanisme redouté — un TFT strict qui punit une défection du LLM — **ne se déclenche pas**, précisément parce que le proxy ne défecte jamais. La prédiction théorique du §2 n'est donc pas observée dans ce jeu : c'est la limite du proxy, et le §7 y reviendra.
3. **`scripted_only`** : coopération **autour de `0.90`** — moyenne `0.9012` sur les cinq graines, écart-type inter-graines `0.0054` (§7) — avec une dispersion intra-graine non nulle (écart-type `0.08` à `0.11`). Le TFT suspect défecte au premier tour, le généreux le copie puis pardonne avec probabilité `0.1`, d'où une convergence vers la coopération **traînée** par un résidu de défection. C'est la seule hétérogénéité dont le résultat est véritablement dynamique, et la seule dont les valeurs dépendent du générateur.
4. **`always_defect`** : **0.0000** partout, payoff `100` partout. Le contrôle négatif passe.

**Ce que la dispersion dit.** L'écart-type par graine est **nul** dans les trois régimes déterministes (`llm_only`, `mix_llm_scripted`, `always_defect`) : le même résultat à chaque rencontre, quel que soit le tirage. Il n'est non nul que pour `scripted_only`. C'est donc la variabilité **entre hétérogénéités** qui porte le signal, et non le bruit des rencontres — exactement ce que le gel §0 anticipe.


## 5. Modèle nul — `uniform_random_seed`

Le **modèle nul** est la baseline inerte : chaque tour, l'agent joue `C` ou `D` avec probabilité 0.5, sans mémoire. Le taux de coopération attendu est **0.5** par construction, et la dispersion observée sur les graines mesure le bruit de fond de l'échantillonnage.

C'est cette dispersion qui justifie la bande de nullité : une hétérogénéité dont la moyenne tombe dans `0.5 ± 0.05` est **indiscernable du hasard** à cette taille d'échantillon.


In [7]:
// §5 — Modele nul : uniform_random_seed (baseline inerte)
var NULL_HET = "uniform_random_seed";   // hors HETEROGENEITIES, sert uniquement aux §5 et §7
var nullResults = new Dictionary<int, List<MatchResult>>();
foreach (var seed in SEEDS)
{
    var rows = new List<MatchResult>();
    for (var m = 0; m < N_MATCHES_PER_PAIR; m++)
        rows.Add(RunMatch("uniform_random_seed", "uniform_random_seed", seed * 1000 + m));
    nullResults[seed] = rows;
}

var nullCoopPerSeed = new Dictionary<int, double>();
foreach (var seed in SEEDS)
    nullCoopPerSeed[seed] = Mean(nullResults[seed].Select(r => r.CoopRate0));

Console.WriteLine("§5 — Modele nul (uniform_random vs uniform_random) :");
Console.WriteLine("  " + "seed".PadLeft(5) + "mean_coop_0".PadLeft(13) + "   (attendu ~ 0.5)");
foreach (var seed in SEEDS)
    Console.WriteLine($"  {seed,5}{nullCoopPerSeed[seed],13:F4}");

var allNulls = SEEDS.Select(s => nullCoopPerSeed[s]).ToList();
var nullMean = Mean(allNulls);
var nullStd = Stdev(allNulls);
Console.WriteLine();
Console.WriteLine($"  Moyenne globale du nul : {nullMean:F4} (attendu ~ 0.5)");
Console.WriteLine($"  Ecart-type inter-seed   : {nullStd:F4}");

// LOI : le nul tombe dans la bande H0, par graine et en moyenne
foreach (var seed in SEEDS)
    Check(Math.Abs(nullCoopPerSeed[seed] - 0.5) <= H0_NULL_BAND,
          $"Modele nul seed={seed} hors bande H0 (+/- {H0_NULL_BAND} autour de 0.5) : {nullCoopPerSeed[seed]:F4}");
Check(Math.Abs(nullMean - 0.5) <= H0_NULL_BAND, $"Modele nul hors bande H0 en moyenne : {nullMean:F4}");
Console.WriteLine($"  Assertion de loi : nul dans la bande +/- {H0_NULL_BAND} de 0.5 (5 seeds + moyenne).");


§5 — Modele nul (uniform_random vs uniform_random) :


   seed  mean_coop_0   (attendu ~ 0.5)


      0       0.5046


      1       0.5056


      7       0.5064


     42       0.5146


     99       0.5038


  Moyenne globale du nul : 0.5070 (attendu ~ 0.5)


  Ecart-type inter-seed   : 0.0044


  Assertion de loi : nul dans la bande +/- 0.05 de 0.5 (5 seeds + moyenne).


**Lecture — §5.** Le modèle nul rend une moyenne proche de **0.5** et l'assertion vérifie qu'elle reste dans la bande `± 0.05` **sur chacune des cinq graines**, puis en moyenne. C'est la référence à laquelle le §7 compare chaque hétérogénéité.

Deux précisions de lecture, qui valent pour les deux carnets. D'abord, la bande est posée autour du nul **mesuré** et non autour de la valeur théorique `0.5` : le nul fluctue lui-même d'une graine à l'autre, et la référence du test doit être la mesure, pas l'idéal. Ensuite, l'écart-type inter-graines du nul estime le **bruit de fond** : c'est lui qui donne son sens à l'écart observé au §7, et non une valeur absolue.


## 6. Contrôle négatif — `always_defect` doit dégrader la coopération à 0 %

Le contrôle négatif est défini au §2. Sa propriété attendue : **taux de coopération exactement 0** (les deux agents jouent `D` à chaque tour) et payoff de `P × N_ROUNDS = 100` par agent (défection mutuelle stable).

**Si cette propriété est violée**, c'est la chaîne de simulation qui est cassée, pas la stratégie. Le contrôle est donc exécuté comme une **assertion**, pas comme un affichage : une violation arrête le carnet avant le verdict.


In [8]:
// §6 — Controle negatif : always_defect doit donner coop = 0% sur tous les seeds
var negViolations = new List<string>();
foreach (var seed in SEEDS)
{
    var a = aggregated["always_defect"][seed];
    if (a.MeanCoop0 != 0.0 || a.MeanCoop1 != 0.0)
        negViolations.Add($"seed={seed} coop=({a.MeanCoop0}, {a.MeanCoop1})");
    if (a.MeanPayoff0 != P * N_ROUNDS)
        negViolations.Add($"seed={seed} payoff={a.MeanPayoff0}");
}
Check(negViolations.Count == 0, "Controle negatif viole : " + string.Join(" | ", negViolations));

var neg0 = aggregated["always_defect"][0];
var neg99 = aggregated["always_defect"][99];
Console.WriteLine("§6 — Controle negatif OK : always_defect -> coop = 0% sur les 5 seeds.");
Console.WriteLine($"  always_defect seed=0  : coop_0 = {neg0.MeanCoop0}, payoff_0 = {neg0.MeanPayoff0}");
Console.WriteLine($"  always_defect seed=99 : coop_0 = {neg99.MeanCoop0}, payoff_0 = {neg99.MeanPayoff0}");
Console.WriteLine($"  Payoff attendu (defection mutuelle) : {P * N_ROUNDS}");


§6 — Controle negatif OK : always_defect -> coop = 0% sur les 5 seeds.


  always_defect seed=0  : coop_0 = 0, payoff_0 = 100


  always_defect seed=99 : coop_0 = 0, payoff_0 = 100


  Payoff attendu (defection mutuelle) : 100


**Lecture — §6.** Le contrôle négatif passe sur les cinq graines : coopération exactement `0`, payoff exactement `100`. La chaîne est donc validée, et les hétérogénéités du §4 deviennent comparables entre elles et au modèle nul.

Ce contrôle a une fonction précise dans la lecture du carnet : il prouve que le protocole **sait produire un zéro**. Sans lui, un zéro observé ailleurs pourrait être un artefact — un historique vide, une politique jamais appelée, un générateur qui ne tire pas. Ici, le zéro est obtenu par les mêmes fonctions que les autres hétérogénéités, ce qui ferme cette hypothèse.


## 7. Verdict H0/H1 par hétérogénéité

**H0** : *toutes les hétérogénéités produisent un taux de coopération moyen dans la bande `± 0.05` du modèle nul*. Une hétérogénéité dans la bande est indiscernable du hasard : pas de coopération attribuable à la composition de la population.

**H1** : *au moins une hétérogénéité sort de la bande* — la composition de la population **importe**.

**Verdict par hétérogénéité** : on moyenne le taux de coopération des deux agents sur les cinq graines, et on compare au nul.


In [9]:
// §7 — Verdict H0/H1 par heterogeneite
var verdicts = new Dictionary<string, (double MeanCoop, double StdCoop, bool InBand, string Verdict)>();
foreach (var het in HETEROGENEITIES)
{
    var coopMeans = new List<double>();
    foreach (var seed in SEEDS)
    {
        var a = aggregated[het][seed];
        coopMeans.Add((a.MeanCoop0 + a.MeanCoop1) / 2);
    }
    var meanCoop = Mean(coopMeans);
    var stdCoop = Stdev(coopMeans);
    var inBand = Math.Abs(meanCoop - nullMean) <= H0_NULL_BAND;
    verdicts[het] = (meanCoop, stdCoop, inBand, inBand ? "H0" : "H1");
}

Console.WriteLine("§7 — Verdict H0/H1 par heterogeneite :");
Console.WriteLine($"  Modele nul (mesure .NET) : mean_coop = {nullMean:F4}, H0 band = +/- {H0_NULL_BAND}");
Console.WriteLine();
Console.WriteLine("  " + "heterogeneite".PadRight(20) + "mean_coop".PadLeft(11)
                  + "std".PadLeft(9) + "verdict".PadLeft(9));
foreach (var het in HETEROGENEITIES)
{
    var v = verdicts[het];
    Console.WriteLine($"  {het,-20}{v.MeanCoop,11:F4}{v.StdCoop,9:F4}{v.Verdict,9}");
}

var h0Count = verdicts.Values.Count(v => v.Verdict == "H0");
var h1Count = verdicts.Values.Count(v => v.Verdict == "H1");
Console.WriteLine();
Console.WriteLine($"  Bilan : {h0Count} H0, {h1Count} H1");
Console.WriteLine("  Lecture : scripted_only et le controle negatif mesurent quelque chose ;");
Console.WriteLine("  llm_only et mix_llm_scripted sont des consequences de l'implementation");
Console.WriteLine("  (aucune defection n'est jouee, la cooperation mutuelle est le point fixe).");

// LOIS attendues du gel §0 (le detail numerique du scripted_only est propre au generateur)
Check(verdicts["llm_only"].MeanCoop == 1.0, "llm_only doit cooperer a 100%.");
Check(verdicts["mix_llm_scripted"].MeanCoop == 1.0, "mix_llm_scripted doit cooperer a 100%.");
Check(verdicts["always_defect"].MeanCoop == 0.0, "always_defect doit cooperer a 0%.");
var scriptedOnly = verdicts["scripted_only"].MeanCoop;
Check(scriptedOnly > nullMean + H0_NULL_BAND,
      $"scripted_only doit sortir de la bande H0 par le haut : {scriptedOnly:F4}");
Check(scriptedOnly < 1.0,
      $"scripted_only doit rester sous 1.0 (defection initiale du TFT suspect) : {scriptedOnly:F4}");
Check(h0Count == 0 && h1Count == 4, $"Attendu 0 H0 / 4 H1, obtenu {h0Count} H0 / {h1Count} H1.");
Console.WriteLine("  Verdict global : la cooperation depend de l'heterogeneite — mesuree sur");
Console.WriteLine("  scripted_only, rappelee par le controle negatif, acquise par construction");
Console.WriteLine("  sur les deux paires a proxy.");


§7 — Verdict H0/H1 par heterogeneite :


  Modele nul (mesure .NET) : mean_coop = 0.5070, H0 band = +/- 0.05


  heterogeneite         mean_coop      std  verdict


  llm_only                 1.0000   0.0000       H1


  mix_llm_scripted         1.0000   0.0000       H1


  scripted_only            0.9012   0.0054       H1


... (4 outputs left out)

  (aucune defection n'est jouee, la cooperation mutuelle est le point fixe).


  Verdict global : la cooperation depend de l'heterogeneite — mesuree sur


  scripted_only, rappelee par le controle negatif, acquise par construction


  sur les deux paires a proxy.


**Lecture — §7.** Les quatre hétérogénéités sortent de la bande de nullité — `0 H0 / 4 H1` — mais ce compte ne vaut pas quatre résultats. **Deux cellules mesurent quelque chose** ; **les deux autres sont des conséquences de l'implémentation**, là où aucune défection n'est jamais jouée et où la coopération mutuelle est le point fixe. C'est la même lecture que le carnet Python, atteinte avec des suites pseudo-aléatoires différentes.

- `llm_only` : **H1 — conséquence de l'implémentation**, coopération `1.0000` avec un écart-type nul. Aucune défection n'est jamais disponible dans cette paire : le verdict est exact et il n'apprend rien sur la dynamique.
- `mix_llm_scripted` : **H1 — conséquence de l'implémentation**, coopération `1.0000`, écart-type nul. Le contraste avec le prédicat du §2 est instructif : le mécanisme de représailles existe dans le code et n'est jamais déclenché, faute de défection à punir.
- `scripted_only` : **H1 — mesure.** Coopération élevée mais strictement inférieure à `1`, avec une dispersion inter-graines non nulle. C'est le seul verdict de coopération dont la valeur dépend du générateur, et le seul où la **dynamique** se voit — l'assertion ne fixe donc pas sa valeur, seulement sa position : au-dessus de la bande, sous le plafond.
- `always_defect` : **H1 — contrôle négatif**, du côté opposé au nul (`0.0000`). Un contrôle négatif classé H1 n'est pas un résultat de coopération : c'est la preuve que l'instrument distingue une population d'une autre. S'il avait été classé H0, la bande aurait déclaré que zéro coopération est un comportement d'appoint, et tous les autres verdicts seraient devenus illisibles.

**Réserve de portée, la même que dans le carnet Python.** Le verdict porte sur des **politiques scriptées**. Le `1.0000` de `llm_only` est la conséquence d'un proxy qui ne déçoit pas dans ce jeu, non une mesure de la coopération d'un modèle de langage. Le protocole est réutilisable tel quel pour un agent réel : il suffit de remplacer le proxy, le gel et les seuils restant inchangés — c'est précisément ce que l'exercice 1 commence à faire.


## Conclusion du protocole

Le carnet étend la sous-série `GameTheory-16` en rejouant un **protocole reproductible de mesure de la coopération** sur l'IPD entre agents hétérogènes. Le cadre repose sur quatre pièces, toutes vérifiables dans le code exécuté :

1. **Gel §0** : quatre hétérogénéités, cinq graines, seuils H0/H1 explicites, modèle nul et contrôle négatif documentés **avant** le premier tour.
2. **Proxy scripté** des agents LLM (`llm_proxy_tft` = `generous_tft`, Vallinder–Hughes 2024) — limite assumée de la version 1, CPU-only.
3. **Multi-seed** sur `{0, 1, 7, 42, 99}` + **contrôle négatif** `always_defect` (zéro strict) + **modèle nul** `uniform_random_seed` (bande autour de 0.5).
4. **Verdict H0/H1** par hétérogénéité, lu contre les seuils du gel et non contre un arbitrage post-hoc.

**Ce que la version C# ajoute.** Les deux carnets partagent le protocole et les lois, pas les suites tirées : le Python affirme ses valeurs avec MT19937, le C# affirme les mêmes lois avec `new Random(seed)`. Un auditeur qui veut vérifier la parité lit donc les **assertions** — bande du nul, zéro du contrôle, coopération totale des paires à proxy, `0 H0 / 4 H1` — et non une égalité de chiffres qui n'a pas de sens entre deux générateurs.

**Déconflit** : pas de réécriture de `16d` (pilote Othman–Sandholm) ni de `16e` (populations) ; pas de recouvrement avec la formalisation Lean Munkres (Coalitions-05) ni avec `16b`/`16c` (conception de mécanismes).


## Écarts d'exécution vs gel

Aucun écart **matériel** entre le gel §0 et l'exécution de ce jumeau :

- `SEEDS = [0, 1, 7, 42, 99]` — vérifié par assertion dans la cellule du §0.
- `N_ROUNDS = 100`, `N_MATCHES_PER_PAIR = 50` — vérifiés par assertion.
- `HETEROGENEITIES` = les quatre du gel, dont `always_defect` — vérifié par égalité d'ensembles au §2.
- `NEGATIVE_CONTROL = always_defect` : zéro coopération et payoff `100` sur les cinq graines — vérifié par assertion au §6.
- `NULL_MODEL = uniform_random_seed` : dans la bande `± 0.05` de 0.5 sur les cinq graines et en moyenne — vérifié par assertion au §5.

**Un écart de nature, consigné** : le générateur. Le protocole gèle les graines, pas l'algorithme pseudo-aléatoire ; le Python utilise MT19937, le C# la suite de la BCL. Toute valeur qui dépend d'un tirage — le nul, `scripted_only`, les écarts-types — diffère donc entre les deux carnets sans que la parité soit en cause. Les grandeurs déterministes (`llm_only` et `mix_llm_scripted` à `1`, `always_defect` à `0`) coïncident exactement, et ce sont elles que les assertions fixent valeur par valeur.

**Coût total** : 100 000 tours IPD, sans GPU et sans appel réseau.


## Exercices

Les trois exercices prolongent le carnet sur ses propres leviers : généraliser le proxy LLM, passer d'un tournoi à une dynamique de population, et introduire un bruit de perception entre agents.


### Exercice 1 — *Politique LLM étendue* (générosité paramétrable)

Le proxy utilise `generous_tft` avec `p_gen = 0.1`. **Question** : la coopération moyenne de `llm_only` dépend-elle significativement de `p_gen` ? Trace la courbe `mean_coop(llm_only)` pour `p_gen ∈ {0.0, 0.05, 0.1, 0.2, 0.3, 0.5}` — du TFT strict (0.0) au généreux extrême (0.5).

Construis une fabrique de politique `GenerousTftWith(double pGen)` qui rend une `Func<List<Round>, Random, int, string>` ayant la même signature que celles du registre, puis réutilise `RunMatch` en substituant la politique dans une copie locale du dictionnaire.


In [10]:
// Exercice 1 — Politique LLM etendue (p_gen variable) — a completer
// Objectif : mesurer mean_coop(llm_only) pour p_gen in {0.0, 0.05, 0.1, 0.2, 0.3, 0.5}.
// Indice : ecris une fabrique GenerousTftWith(pGen) qui rend la meme signature que
//          les politiques du registre, puis substitue-la dans une copie de POLICIES.
// Indice 2 : pour chaque p_gen, agrege sur les 5 seeds et affiche la courbe.
// TODO etudiant
Console.WriteLine("Exercice 1 a completer.");


Exercice 1 a completer.


### Exercice 2 — *Dynamique évolutive de population* (reproduction et mutation)

En IPD, la dynamique évolutive (Axelrod 1984) simule des **générations successives** où les stratégies les plus performantes se reproduisent, avec un faible taux de mutation. **Question** : la population converge-t-elle vers TFT, vers le TFT généreux, ou oscille-t-elle ?

Pars d'une population initiale mixte (50 % TFT, 50 % `always_defect`) et itère 50 générations : à chaque génération, chaque agent joue l'IPD contre plusieurs adversaires, les stratégies se reproduisent proportionnellement au payoff cumulé, et une mutation de 5 % remplace une stratégie par une autre.


In [11]:
// Exercice 2 — Dynamique evolutive — a completer
// Objectif : population initiale mixte (50% TFT, 50% always_defect), 50 generations.
// Indice : a chaque generation, (1) chaque agent joue l'IPD contre N adversaires,
//   (2) les strategies sont proportionnelles au payoff cumule (reproduction),
//   (3) un faible taux de mutation (5%) remplace une strategie par une autre.
// TODO etudiant
Console.WriteLine("Exercice 2 a completer.");


Exercice 2 a completer.


### Exercice 3 — *Bruit de communication* (canal symétrique bruité)

En IPD réel, deux agents ne connaissent pas toujours le coup joué par l'adversaire : bruit de perception, latence, erreur de transmission. **Question** : la coopération moyenne de `llm_only` est-elle robuste à un bruit de perception `p_noise ∈ {0.0, 0.05, 0.1, 0.2}`, où le coup observé est inversé avec probabilité `p_noise` ?

L'indice décisif est que le bruit porte sur l'**observation**, pas sur le coup joué : la politique reçoit un historique dont l'entrée adverse a été éventuellement inversée, tandis que le payoff reste calculé sur les coups réels.


In [12]:
// Exercice 3 — Bruit de communication — a completer
// Objectif : mesurer mean_coop(llm_only) pour p_noise in {0.0, 0.05, 0.1, 0.2}.
// Indice : ajoute une couche de bruit entre l'historique reel et l'historique observe
//          que lit la politique ; le payoff reste calcule sur les coups reels.
// Indice 2 : une politique noisy_tft(noise) observe l'historique inverse.
// TODO etudiant
Console.WriteLine("Exercice 3 a completer.");


Exercice 3 a completer.


## À retenir

Ce carnet mesure la **coopération** entre agents dans le dilemme du prisonnier itéré, sous un protocole gelé au §0 : la tentation `T = 5`, la récompense de la coopération mutuelle `R = 3`, la sanction `P = 1` et le coût de la dupe `S = 0`, avec `2R > T + S` (`6 > 5`) — la condition qui rend la coopération rentable sur la durée. Le LLM n'est pas appelé : il est remplacé par un **proxy scripté** (`generous_tft`), calibré sur Vallinder et Hughes (2024). Ce que le carnet mesure est le protocole et le contraste entre hétérogénéités, pas le comportement d'un modèle de langage réel.

**Ce que le protocole établit.**

- **L'échelle** : 100 tours par rencontre, 50 rencontres par paire, 5 graines (`[0, 1, 7, 42, 99]`), 4 hétérogénéités définies comme des couples de politiques.
- **Le modèle nul** : `uniform_random_seed` produit une coopération centrée sur `0.5` — dans la bande `± 0.05` sur chacune des cinq graines, ce que l'assertion du §5 vérifie.
- **Le contrôle négatif** : `always_defect` donne exactement `0` de coopération et un payoff de `100` sur les cinq graines — le plancher de l'échelle, vérifié par assertion.
- **Les quatre verdicts** : `llm_only` et `mix_llm_scripted` à `1`, `scripted_only` au-dessus de la bande et sous le plafond, `always_defect` à `0` — soit **0 H0 / 4 H1**, avec la bande posée avant le run. Sur ces quatre, `scripted_only` mesure la dynamique et `always_defect` établit que l'instrument distingue deux populations ; les deux paires à proxy, elles, sont acquises par construction.

**Réserve de portée.** Le verdict global — « la coopération dépend de l'hétérogénéité » — porte sur des **politiques scriptées**. Le taux de `1` de `llm_only` est la conséquence d'un proxy qui ne déçoit pas dans ce jeu, non une mesure de la coopération d'un modèle de langage. Le protocole est réutilisable tel quel pour un agent réel : il suffit de remplacer le proxy, le gel et les seuils restant inchangés.

## Vérifiez votre compréhension

1. Les deux hétérogénéités à proxy atteignent `1.0000` avec un écart-type nul. Pourquoi l'assertion du §7 vérifie-t-elle cette valeur exactement, alors qu'elle n'impose aucune valeur à `scripted_only` ?

   <details><summary>Réponse</summary>

   Parce que les deux cas n'ont pas le même statut. Dans `llm_only` et `mix_llm_scripted`, aucune défection n'est jamais jouée : la branche de pardon de `generous_tft` n'est pas atteinte, aucun tirage n'a lieu, et le résultat est **indépendant du générateur** — il vaut `1` par construction, et cela doit être vrai des deux côtés de la parité. Dans `scripted_only`, la trajectoire dépend des pardons tirés, donc du générateur : fixer la valeur reviendrait à affirmer une égalité qui n'existe pas entre MT19937 et la BCL. L'assertion y fixe donc la **position** — au-dessus de la bande de nullité, strictement sous `1` — et non le chiffre.

   </details>

2. Le modèle nul est asserté dans la bande `± 0.05` de `0.5`. Pourquoi une bande aussi large, alors que la moyenne observée est proche de `0.5` à quelques millièmes ?

   <details><summary>Réponse</summary>

   La bande n'est pas un intervalle de confiance, c'est le **seuil du test** : elle définit ce que « indiscernable du hasard » veut dire pour ce protocole. Elle est volontairement large pour que le test ne dépende ni du générateur ni de la graine — un seuil serré aurait transformé le bruit d'échantillonnage en faux signal. La dispersion réelle du nul est bien plus petite, et c'est justement elle qui rend les écarts du §7 lisibles : `scripted_only` comme `llm_only` sortent de la bande par une marge qui n'a rien à voir avec le bruit.

   </details>

3. `always_defect` est classé **H1** avec un taux de `0`. Un contrôle négatif peut-il être un résultat H1 ?

   <details><summary>Réponse</summary>

   Oui, et c'est le résultat attendu : H1 signifie « le comportement s'écarte de la bande de nullité », et le contrôle négatif doit s'en écarter **du côté opposé**. C'est ce qui prouve que l'instrument distingue une population d'une autre. S'il était classé H0, la bande aurait déclaré que zéro coopération est un comportement d'appoint, et les autres verdicts du run deviendraient illisibles. Le H1 du contrôle négatif est un H1 **de dérive**, pas de coopération.

   </details>

4. `mix_llm_scripted` atteint `1` alors que le §2 prédit qu'un TFT strict punirait la moindre défection du LLM. La prédiction est-elle fausse ?

   <details><summary>Réponse</summary>

   Elle n'est pas fausse, elle est **inapplicable à ce proxy**. Le mécanisme de représailles existe bien dans le code : si le LLM défectait, le TFT strict copierait la défection au tour suivant. Mais le proxy `generous_tft` ne défecte jamais tant qu'il est copié, donc la condition d'entrée du mécanisme ne se produit pas. C'est une limite du proxy, pas une réfutation du modèle — et c'est exactement ce que l'exercice 1 explore, en faisant varier la générosité jusqu'à `0`, où le mécanisme devient observable.

   </details>


## Pour aller plus loin

| # | Référence | Couverture |
|---|-----------|------------|
| 1 | Axelrod R. (1984) *The Evolution of Cooperation* — Basic Books | IPD canonique, tournoi informatique, TFT |
| 2 | Axelrod R., Hamilton W. D. (1981) *Science* 211(4489) — *The Evolution of Cooperation* | Article fondateur, IPD biologique |
| 3 | Nowak M., Sigmund K. (1993) *Nature* 364(6432) 56-58, DOI 10.1038/364056a0 — *A strategy of win-stay, lose-shift that outperforms tit-for-tat in the Prisoner's Dilemma game* | Generous TFT, dynamique évolutive, WSLS |
| 4 | Imhof L. A., Fudenberg D., Nowak M. A. (2005) *PNAS* 102(31) 10797-10800, DOI 10.1073/pnas.0502589102 — *Evolutionary cycles of cooperation and defection* | Cycles de coopération, win-stay lose-shift |
| 5 | Press W. H., Dyson F. J. (2012) *PNAS* 109(26) — *Iterated Prisoner's Dilemma contains strategies that dominate any evolutionary opponent* | Stratégies zéro-déterminant, extension de TFT |
| 6 | **Vallinder A., Hughes C. (2024)** — *Cultural Evolution of Cooperation among LLM Agents* | Proxy LLM, tournoi IPD, générosité des LLM |
| 7 | **Payne J., Alloui M., Cros A. (2025)** — *Strategic Intelligence in Large Language Models* | Intelligence stratégique, populations LLM, hétérogénéité |
| 8 | Sigmund K. (2010) *The Calculus of Selfishness* — Princeton University Press | Synthèse de théorie des jeux évolutionnaires |

**Sources primaires** (gisement `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\`) :

- `2024 - Vallinder Hughes - Cultural Evolution of Cooperation among LLM Agents.pdf`
- `2025 - Payne Alloui-Cros - Strategic Intelligence in Large Language Models.pdf`

**Mapping acquis vers SOTA** (5 acquis du carnet, 8 références) :

1. **IPD canonique** : Axelrod 1984, Axelrod-Hamilton 1981, Sigmund 2010.
2. **Politiques TFT et WSLS** (strict, généreux, suspect, win-stay lose-shift) : Nowak-Sigmund 1993, Imhof-Fudenberg-Nowak 2005.
3. **Stratégies zéro-déterminant** (au-delà de TFT) : Press-Dyson 2012.
4. **Proxy LLM et générosité** : Vallinder-Hughes 2024 (source primaire).
5. **Hétérogénéité de population** : Payne-Alloui-Cros 2025 (source primaire).

**Carnets voisins.**

- [GameTheory-16i — Appariage stable (Gale-Shapley)](GameTheory-16i-Appariage-Stable-Gale-Shapley-CSharp.ipynb) : l'autre axe de la famille 16 — là où ce carnet mesure la **coopération** entre agents sous protocole multi-seed, le 16i mesure ce qu'un mécanisme d'appariage promet à chaque côté.
- [GameTheory-16e — Populations Othman-Sandholm](GameTheory-16e-Populations-Othman-Sandholm-Python.ipynb) : le voisin le plus proche, même exigence de gel pré-enregistré et de contrôle négatif, mais sur l'**incitation** plutôt que sur la coopération.
- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-CSharp.ipynb) : le carnet owner, le cadre de conception de mécanismes et le théorème d'impossibilité qui borne les promesses d'incitation.
